# CHAPTER 6: PHÂN TÍCH VÀ DỰ BÁO CHUỖI THỜI GIAN

## Thực hiện phân tích chuỗi thời gian đối với tỷ lệ thất nghiệp hàng tháng của Mỹ được tải từ Nasdaq Data Link.

### 1. Nhập thư viện và xác thực:

In [ ]:
import pandas as pd
import nasdaqdatalink
import seaborn as sns
from statsmodels.tsa.stattools import seasonal_decompose

nasdaqdatalink.ApiConfig.api_key = "YOUR_API_KEY"

### 2. Tải số liệu từ năm 2010 đến năm 2019:

In [ ]:
df = (
    nasdaqdatalink.get("FRED/UNRATE", 
                       start_date="2010-01-01", 
                       end_date="2019-12-31")
    .rename(columns={"Value": "unmep_rate"})
)

### 3. Cộng giá trị trung bình trượt và độ lệch chuẩn:

In [ ]:
WINDOW_SIZE = 12 
df["rolling_mean"] = df["unmep_rate"].rolling(window=WINDOW_SIZE).mean()
df["rolling_std"] = df["unmep_rate"].rolling(window=WINDOW_SIZE).std()
df.plot(title="Unemployment rate")

### 4. Tiến hành phân tích thành phần theo mùa bằng mô hình cộng:

In [ ]:
decomposition_result = seasonal_decompose(df['unemp_rate'], model='additive')

(
    decomposition_results
    .plot()
    .suptitle('Additive Decomposition')
)

### Phân tích bằng phương pháp `STL`

In [ ]:
from statsmodels.tsa.seasonal import STL

stl_decomposition = STL(df[['unemp_rate']]).fit()
stl_decomposition.plot() \
                 .suptitle('STL Decomposition') 

## Kiểm tra chuỗi thời gian tỷ lệ thất nghiệp hàng tháng có ổn định hay không

### 1. Nhập thư viện:

In [ ]:
import pandas as pd
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller,kpss

### 2. Định nghĩa một hàm để chạy bài kiểm tra ADF:

In [ ]:
def adf_test(x):
    indices = ['Test Statistic', 'p-value', '# of Lags Used', 'Number of Observations Used']

    adf_test = adfuller(x, autolag='AIC')
    results = pd.Series(adf_test[0:4], index=indices)

    for key, value in adf_test[4].items():
        results[f'Critical Value ({key})'] = value

    return results

In [ ]:
adf_test(df['unmep_rate'])

### 3. Định nghĩa một hàm để chạy thử nghiệm KPSS:

In [ ]:
def kpss_test(x, h0_type='c'):
    indices = ['Test Statistic', 'p-value', '# of Lags Used']

    kpss_test = kpss(x, regression='h0_type')
    results = pd.Series(kpss_test[0:3], index=indices)

    for key, value in kpss_test[3].items():
        results[f'Critical Value ({key})'] = value

    return results

In [ ]:
kpss_test(df['unmep_rate'])

### 4. Tạo biểu đồ ACF/PACF:

In [ ]:
N_LAGS = 40
SIGNIFICANCE_LEVEL = 0.05

fig, ax = plt.subplots(2, 1)
plot_acf(df['unmep_rate'], lags=N_LAGS, ax=ax[0],alpha=SIGNIFICANCE_LEVEL)
plot_pacf(df['unmep_rate'], lags=N_LAGS, ax=ax[1],alpha=SIGNIFICANCE_LEVEL)

### Kiểm tra ADF bằng thư viện `arch`:

In [ ]:
from arch.unitroot import ADF
adf = ADF(df['unmep_rate'])
print(adf.summary().as_text())

### Bài kiểm tra Zivot-Andrews:

In [ ]:
from arch.uniroot import ZivotAndrews
za = ZivotAndrews(df["unemp_rate"])
print(za.summary().as_text())

## Chuyển đổi chuỗi từ không ổn định sang ổn định:

### 1. Nhập các thư viện, xác thực và cập nhật dữ liệu lạm phát:

In [ ]:
import pandas as pd
import numpy as np
import nasdaqdatalink
import cpi 
from datatime import date
from chapter_6_utils import test_autocorrelation

nasdaqdatalink.ApiConfig.api_key = "YOUR_API_KEY"